In [1]:
import torch
import numpy as np
import pandas as pd
from torchvision import models, transforms
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import requests
import os
device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = torch.nn.Identity()
model = model.to(device)
model.eval()

print("Using device:", device)


Using device: cpu


In [2]:
#creating poster urls
def get_tmdb_poster_url(poster_path, size="w500"):
    if poster_path is None or str(poster_path).strip() == "":
        return None
    return f"https://image.tmdb.org/t/p/{size}{poster_path}"


In [3]:
movies=pd.read_csv("D:\MINI PROJECT\DATASET\movies_final.csv")

In [4]:
#poster extraction
movies = pd.read_csv(r"D:\MINI PROJECT\DATASET\movies_final.csv")
num_movies = len(movies)
print("Total movies:", num_movies)


Total movies: 23138


In [5]:
#preprocessing and embedding functions
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def extract_poster_embedding(image):
    image_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        features = model(image_tensor)

    return features.squeeze().cpu().numpy()


In [6]:
# %%
from torchvision.models.detection import fasterrcnn_resnet50_fpn

# Load object detector for region attention (NOVEL)
obj_detector = fasterrcnn_resnet50_fpn(weights='COCO_V1').to(device).eval()

def extract_region_embedding(image):
    """NEW: Region-attended embedding (faces/text regions)"""
    detections = obj_detector(preprocess(image).unsqueeze(0).to(device))[0]
    boxes = detections['boxes']
    
    if len(boxes) == 0:
        return extract_poster_embedding(image)  # Fallback to global
    
    region_features = []
    for box in boxes[:3]:  # Top 3 regions
        x1,y1,x2,y2 = box[:4].cpu().numpy()
        if x2-x1 > 50 and y2-y1 > 50:  # Valid size
            region = image.crop((int(x1),int(y1),int(x2),int(y2)))
            region_tensor = preprocess(region).unsqueeze(0).to(device)
            region_features.append(model(region_tensor).squeeze())
    
    if region_features:
        return torch.stack(region_features).mean(0).cpu().numpy()
    return extract_poster_embedding(image)

print("✅ Region attention loaded!")


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to C:\Users\Admin/.cache\torch\hub\checkpoints\fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [02:12<00:00, 1.27MB/s] 


✅ Region attention loaded!


In [7]:
#resume safe initialization
EMBED_PATH = "poster_embeddings.npy"
STATUS_PATH = "poster_status.npy"

EMB_DIM = 2048

if os.path.exists(EMBED_PATH) and os.path.exists(STATUS_PATH):
    poster_embeddings = np.load(EMBED_PATH)
    status = np.load(STATUS_PATH)
    print("Resuming extraction")
else:
    poster_embeddings = np.zeros((num_movies, EMB_DIM), dtype=np.float32)
    status = np.zeros(num_movies, dtype=np.int8)  # 0 = not done, 1 = success, -1 = failed
    print("Starting fresh")

print("Already done:", np.sum(status == 1))
print("Already failed:", np.sum(status == -1))


Starting fresh
Already done: 0
Already failed: 0


In [8]:
EMBED_PATH = r"D:\MINI PROJECT\NOTEBOOKS\poster_embeddings.npy"
STATUS_PATH = r"D:\MINI PROJECT\NOTEBOOKS\poster_status.npy"
num_movies = len(movies)
EMB_DIM = 2048

if os.path.exists(EMBED_PATH) and os.path.exists(STATUS_PATH):
    poster_embeddings = np.load(EMBED_PATH)
    status = np.load(STATUS_PATH)
    print("Resuming extraction")
else:
    poster_embeddings = np.zeros((num_movies, EMB_DIM), dtype=np.float32)
    status = np.zeros(num_movies, dtype=np.int8)
    print("Starting fresh")

print("Already extracted:", np.sum(status == 1))
print("Already failed:", np.sum(status == -1))


Starting fresh
Already extracted: 0
Already failed: 0


In [9]:
failed = int((status == -1).sum())

for i in tqdm(range(num_movies), desc="Extracting poster embeddings"):
    if status[i] != 0:
        continue

    try:
        poster_path = movies.loc[i, "poster_path"]
        url = get_tmdb_poster_url(poster_path)

        if url is None:
            raise ValueError("No poster")

        response = requests.get(url, timeout=10)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content)).convert("RGB")
        emb = extract_region_embedding(image)  # ← NEW LINE (region attention)
        poster_embeddings[i] = emb

        status[i] = 1

    except Exception:
        status[i] = -1
        failed += 1

    if i % 100 == 0:
        np.save(EMBED_PATH, poster_embeddings)
        np.save(STATUS_PATH, status)

np.save(r"D:\MINI PROJECT\NOTEBOOKS\region_poster_embeddings.npy", poster_embeddings)
np.save(STATUS_PATH, status)
print("✅ Region-attended embeddings saved!")


print("Done.")
print("Failed:", failed)


Extracting poster embeddings: 100%|██████████| 23138/23138 [17:21:24<00:00,  2.70s/it]       


✅ Region-attended embeddings saved!
Done.
Failed: 22998


In [10]:
norms = np.linalg.norm(poster_embeddings, axis=1)

print("Min norm:", norms.min())
print("Mean norm:", norms.mean())
print("Zero vectors:", np.sum(norms == 0))
print("Successful embeddings:", np.sum(status == 1))


Min norm: 0.0
Mean norm: 0.08855607
Zero vectors: 22998
Successful embeddings: 140


In [13]:
import numpy as np

old_emb = np.load("D:\MINI PROJECT\DATASET\DataBackup\poster_embeddings.npy")
old_status = np.load("poster_status.npy")

print("Total vectors:", old_emb.shape)
print("Successful:", np.sum(old_status == 1))

# Check they're not zero vectors
norms = np.linalg.norm(old_emb, axis=1)
print("Non-zero embeddings:", np.sum(norms > 0))

Total vectors: (23138, 2048)
Successful: 140
Non-zero embeddings: 9056


In [14]:
# Make sure the non-zero ones are spread across movies, not clustered
nonzero_indices = np.where(norms > 0)[0]
print("First few indices with embeddings:", nonzero_indices[:10])
print("Last few:", nonzero_indices[-10:])
print("Coverage: {:.1f}% of movies have posters".format(
    len(nonzero_indices) / len(old_emb) * 100
))

First few indices with embeddings: [0 1 2 3 4 5 6 7 8 9]
Last few: [23128 23129 23130 23131 23132 23133 23134 23135 23136 23137]
Coverage: 39.1% of movies have posters
